# Experiments for Intent Classification on the Labeled Dataset

## Evaluate Included Predictions

Start Jupyter from `src` so the relative data paths and sibling imports resolve. Run the following cell independently to evaluate all released prediction files using the published folds. It does not train models, download weights, or contact OpenAI, Ollama, or Rasa.

The remaining sections reproduce individual experiments. Embedding baselines download encoder weights; local LLMs require the original serving setup; GPT experiments require an OpenAI key and incur API costs. Cells with `prompt_variant="rasa"` additionally require the external trained Rasa model and custom ranking service, which are not included. Run only the sections whose dependencies are available.

In [ ]:
from pathlib import Path
from utils import evaluate_predictions, evaluate_skill_predictions

released_folds = Path("../data/dataset_cv.xlsx")
released_predictions = Path("../results/predictions")

for prediction_file in sorted(released_predictions.glob("*.csv")):
    print(f"\n=== {prediction_file.name} ===")
    if prediction_file.name.startswith("binary_"):
        evaluate_predictions(str(released_folds), str(prediction_file))
    elif prediction_file.name.startswith("skills_"):
        evaluate_skill_predictions(str(released_folds), str(prediction_file))

## Inter Annotator Agreement

In [ ]:
"""
The 'label' column (junior researcher) is excluded from the calculation.
"""

from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from sklearn.metrics import cohen_kappa_score

FILE =  "../data/dataset_inter_annotator.xlsx"
FULL_DATASET = FILE 

LABEL_COLS = ["Annotator 1", "Annotator 2", "Annotator 3"]
TARGET_MARGIN = 0.05
N_BOOTSTRAP   = 10_000

# ── load the 350 common rows (all annotators filled) ─────────────────────────
df      = pd.read_excel(FILE, index_col=0, sheet_name='Sheet1')
ratings = df[LABEL_COLS].dropna()   # rows 1–350

all_labels = sorted(set(ratings.values.ravel()))
label2idx  = {lbl: i for i, lbl in enumerate(all_labels)}
codes      = ratings.apply(lambda col: col.map(label2idx)).to_numpy()

# ── Fleiss' kappa ─────────────────────────────────────────────────────────────
table, _ = aggregate_raters(codes)
kappa    = fleiss_kappa(table)

if   kappa < 0:    interp = "poor (worse than chance)"
elif kappa < 0.20: interp = "slight"
elif kappa < 0.40: interp = "fair"
elif kappa < 0.60: interp = "moderate"
elif kappa < 0.80: interp = "substantial"
else:              interp = "almost perfect"

print(f"Fleiss' kappa across {len(LABEL_COLS)} annotators, {len(ratings)} items, {len(all_labels)} categories:")
print(f"  κ = {kappa:.4f}")
print(f"  Interpretation: {interp} agreement (Landis & Koch 1977)")
print()

# ── Pairwise Cohen's kappa ────────────────────────────────────────────────────
pairs = [(LABEL_COLS[i], LABEL_COLS[j])
         for i in range(len(LABEL_COLS)) for j in range(i + 1, len(LABEL_COLS))]
print("Pairwise Cohen's kappa:")
for a, b in pairs:
    k = cohen_kappa_score(ratings[a], ratings[b])
    print(f"  {a:20s} vs {b:20s}  κ = {k:.4f}")
print()

# ── Bootstrap 95% CI ─────────────────────────────────────────────────────────
rng = np.random.default_rng(42)
boot_kappas = np.array([
    fleiss_kappa(aggregate_raters(codes[rng.integers(0, len(codes), len(codes))])[0])
    for _ in range(N_BOOTSTRAP)
])

ci_lo, ci_hi = np.percentile(boot_kappas, [2.5, 97.5])
margin       = (ci_hi - ci_lo) / 2
se           = boot_kappas.std()

print(f"Bootstrap 95% CI (n={len(ratings)} annotated items, {N_BOOTSTRAP:,} resamples):")
print(f"  κ = {kappa:.4f} ± {margin:.4f}  [{ci_lo:.4f}, {ci_hi:.4f}]")
print()

# ── Extrapolated CI for full dataset ─────────────────────────────────────────
FULL_N     = len(pd.read_excel(FULL_DATASET, index_col=0, sheet_name='Sheet1'))
se_full    = se * np.sqrt(len(ratings) / FULL_N)
margin_full = 1.96 * se_full

print(f"Extrapolated 95% CI if full dataset annotated (n={FULL_N:,}):")
print(f"  κ = {kappa:.4f} ± {margin_full:.4f}  [{kappa - margin_full:.4f}, {kappa + margin_full:.4f}]")
print()

# ── Samples needed for target margin ─────────────────────────────────────────
n_needed = int(np.ceil(len(ratings) * (margin / TARGET_MARGIN) ** 2))
print(f"Samples needed for ±{TARGET_MARGIN} margin: {n_needed}")

## Prerequisites

In [ ]:
from train_classifier import train_model, train_model_skill_specific
from predict import run_llm_evaluation_binary, run_llm_evaluation_skills
from utils import save_skill_predictions_to_csv,save_skill_llm_predictions_to_csv, evaluate_predictions, evaluate_skill_predictions
import os
import pandas as pd

DATASET_PATH = "../data/dataset.xlsx"
DATASET_PATH_CV = "../data/dataset_cv.xlsx"

if not os.path.exists(DATASET_PATH_CV):
    from sklearn.model_selection import StratifiedKFold
    
    df = pd.read_excel(DATASET_PATH, sheet_name="Data")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    df['cv_fold'] = -1
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):
        df.loc[val_idx, 'cv_fold'] = fold
    df.to_excel(DATASET_PATH_CV, sheet_name="Data", index=False)

## 1. Binary Classification (intended - unintended)

### Baseline (Encoder + Trained Classifier)

#### Logistic Regression

In [ ]:
import pandas as pd

results = train_model(
  data_path=DATASET_PATH_CV,
  text_col="resolved_query",
  label_col="label",
  split_col="cv_fold",
  id_col="id",
  model_name="intfloat/multilingual-e5-large-instruct",
  model_type="logistic_regression",
  seed=42,
  batch_size=64,
  C=10.0,
  class_weight="balanced",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_logistic_regression_e5.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_logistic_regression_e5.csv")

In [ ]:
import pandas as pd

results = train_model(
  data_path=DATASET_PATH_CV,
  text_col="resolved_query",
  label_col="label",
  id_col="id",
  split_col="cv_fold",
  model_name="BAAI/bge-m3",
  model_type="logistic_regression",
  seed=42,
  batch_size=64,
  C=10.0,
  class_weight="balanced",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_logistic_regression_bge.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_logistic_regression_bge.csv")

#### SVM

In [ ]:
import pandas as pd

results = train_model(
  data_path=DATASET_PATH_CV,
  text_col="resolved_query",
  label_col="label",
  split_col="cv_fold",
  id_col="id",
  model_name="intfloat/multilingual-e5-large-instruct",
  model_type="svm",
  seed=42,
  batch_size=64,
  C=10.0,
  class_weight="balanced",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_svm_e5.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_svm_e5.csv")

In [ ]:
import pandas as pd

results = train_model(
  data_path=DATASET_PATH_CV,
  text_col="resolved_query",
  label_col="label",
  split_col="cv_fold",
  id_col="id",
  model_name="BAAI/bge-m3",
  model_type="svm",
  seed=42,
  batch_size=64,
  C=10.0,
  class_weight="balanced",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_svm_bge.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_svm_bge.csv")

#### Random Forest

In [ ]:
import pandas as pd

results = train_model(
  data_path=DATASET_PATH_CV,
  text_col="resolved_query",
  label_col="label",
  split_col="cv_fold",
  id_col="id",
  model_name="intfloat/multilingual-e5-large-instruct",
  model_type="random_forest",
  seed=42,
  batch_size=64,
  C=1.0,
  class_weight="balanced",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_random_forest_e5.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_random_forest_e5.csv")

In [ ]:
import pandas as pd

results = train_model(
  data_path=DATASET_PATH_CV,
  text_col="resolved_query",
  label_col="label",
  split_col="cv_fold",
  id_col="id",
  model_name="BAAI/bge-m3",
  model_type="random_forest",
  seed=42,
  batch_size=64,
  C=1.0,
  class_weight="balanced",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_random_forest_bge.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_random_forest_bge.csv")

### Language Models

#### Prompts

In [ ]:
ZERO_PROMPT = """You are a classifier for a German voice assistant.

Task:
Classify whether the user utterance is clearly intended for the assistant.

Labels:
1 = intended (direct command, request, or question directed at the assistant)
0 = unintended (background speech, self-talk, conversation, filler, or noise)

Output:
Return exactly one label (0 or 1). If unsure, return 0."""

In [ ]:
FEW_PROMPT = """You are a classifier for a German voice assistant.

Task:
Classify whether the user utterance is clearly intended for the assistant.

Labels:
1 = intended (direct command, request, or question directed at the assistant)
0 = unintended (background speech, self-talk, conversation, filler, or noise)

Examples:
1:
- Leiser.
- Regnet es morgen in München?
- Wie weit habe ich mein Trinkziel schon erreicht?
0:
- Tochter.
- Hast du auch so eine.
- Machst du das ja ich bin sogar bei.
- Die erste kommerzielle Landung auf dem Himmelskörper.
- Innen ist der Zug nach Europa eigentlich schon ab nein.

Output:
Return exactly one label (0 or 1). If unsure, return 0."""

#### Gemma4

In [ ]:
results = run_llm_evaluation_binary(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=ZERO_PROMPT,
    model_name="gemma4:e4b-mlx",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_gemma4_zero.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_gemma4_zero.csv")

In [ ]:
results = run_llm_evaluation_binary(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=FEW_PROMPT,
    model_name="gemma4:e4b-mlx",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_gemma4_few.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_gemma4_few.csv")

#### Qwen 3.5

In [ ]:
results = run_llm_evaluation_binary(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=ZERO_PROMPT,
    model_name="qwen3.5:4b-mlx",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_qwen4b_zero.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_qwen4b_zero.csv")

In [ ]:
results = run_llm_evaluation_binary(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=FEW_PROMPT,
    model_name="qwen3.5:4b-mlx",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_qwen4b_few.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_qwen4b_few.csv")

#### GPT-4.1 Nano

In [ ]:
results = run_llm_evaluation_binary(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=ZERO_PROMPT,
    model_name="gpt-4.1-nano-2025-04-14",
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_gpt-4-1-nano_zero.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_gpt-4-1-nano_zero.csv")

In [ ]:
results = run_llm_evaluation_binary(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=FEW_PROMPT,
    model_name="gpt-4.1-nano-2025-04-14"
)

preds_df = pd.DataFrame.from_dict(results, orient="index")
preds_df.to_csv("../results/predictions/binary_gpt-4-1-nano_few.csv")

evaluate_predictions(DATASET_PATH_CV,"../results/predictions/binary_gpt-4-1-nano_few.csv")

## 2. Skill-Specific Classification (specific skill / out_of_scope / unclear)

### Baseline (Encoder + Trained Classifier)

#### Logistic Regression

In [ ]:
results= train_model_skill_specific(
  data_path=DATASET_PATH_CV,
  text_col="resolved_query",
  label_col="label",
  id_col="id",
  split_col="cv_fold",
  model_name="BAAI/bge-m3",
  model_type="logistic_regression",
  seed=42,
  batch_size=64,
  C=10.0,
  class_weight="balanced",
)

save_skill_predictions_to_csv(results, "../results/predictions/skills_logistic_regression_bge.csv")
evaluate_skill_predictions(DATASET_PATH_CV, "../results/predictions/skills_logistic_regression_bge.csv")

In [ ]:
results= train_model_skill_specific(
  data_path=DATASET_PATH_CV,
  text_col="resolved_query",
  label_col="label",
  id_col="id",
  split_col="cv_fold",
  model_name="intfloat/multilingual-e5-large-instruct",
  model_type="logistic_regression",
  seed=42,
  batch_size=64,
  C=10.0,
  class_weight="balanced",
)

save_skill_predictions_to_csv(results, "../results/predictions/skills_logistic_regression_e5.csv")
evaluate_skill_predictions(DATASET_PATH_CV, "../results/predictions/skills_logistic_regression_e5.csv")

### Language Models

#### Prompts

In [ ]:
ZERO_PROMPT_SKILL = """You are a classifier for a German voice assistant.

Task:
Classify the user utterance into one matching skill label, or use a fallback label if no skill label applies.

Skill labels:
- active_listener: Aktiver Zuhörer / Smalltalk
- blood_pressure_log: Blutdruckmessung protokollieren
- calling: Jemanden anrufen
- change_volume: Lautstärke ändern (lauter / leiser)
- emergency: Notfall auslösen
- general_reminder: Allgemeine Erinnerung stellen
- hydro: Trinkprotokoll
- knowledge_search: Wissensfrage beantworten
- messaging: Nachricht senden
- news_search: Aktuelle Nachrichten abfragen
- notebook: Notiz aufnehmen
- tell_time: Uhrzeit nennen
- time_reminder: Zeitbasierte Erinnerung / Wecker stellen
- vital_measure: Vitalzeichen messen (Blutdruck / Sauerstoff)
- weather: Wetter abfragen
- word_riddle: Worträtsel (alte Schätzchen)
- writing: Witz oder kreativen Text erzählen

Fallback labels:
- out_of_scope (clear user request, but no matching skill is available)
- unclear (intent is ambiguous, incomplete, contradictory, or too noisy to classify)

Output:
Return exactly one label and nothing else. If unsure, return 'unclear'.
"""

In [ ]:
RASA_PROMPT_SKILL = """You are a classifier for a German voice assistant.

Task:
Classify the user utterance into one matching skill label, or use a fallback label if no skill label applies.

Skill labels:
{RASA_INTENTS}

Fallback labels:
- out_of_scope (clear user request, but no matching skill is available)
- unclear (intent is ambiguous, incomplete, contradictory, or too noisy to classify)

Output:
Return exactly one label and nothing else. If unsure, return 'unclear'.
"""

#### Gemma 4

In [ ]:
results = run_llm_evaluation_skills(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=ZERO_PROMPT_SKILL,
    model_name="gemma4:e4b-mlx",
)
save_skill_llm_predictions_to_csv(results, "../results/predictions/skills_gemma4_default.csv")
evaluate_skill_predictions(DATASET_PATH_CV, "../results/predictions/skills_gemma4_default.csv")

In [ ]:

results = run_llm_evaluation_skills(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=RASA_PROMPT_SKILL,
    model_name="gemma4:e4b-mlx",
    prompt_variant="rasa",
)

save_skill_llm_predictions_to_csv(results, "../results/predictions/skills_gemma4_rasa.csv")
evaluate_skill_predictions(DATASET_PATH_CV, "../results/predictions/skills_gemma4_rasa.csv")

#### Qwen 3.5

In [ ]:
results = run_llm_evaluation_skills(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=ZERO_PROMPT_SKILL,
    model_name="qwen3.5:4b-mlx",
)

save_skill_llm_predictions_to_csv(results, "../results/predictions/skills_qwen4b_default.csv")
evaluate_skill_predictions(DATASET_PATH_CV, "../results/predictions/skills_qwen4b_default.csv")

In [ ]:
results = run_llm_evaluation_skills(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=RASA_PROMPT_SKILL,
    model_name="qwen3.5:4b-mlx",
    prompt_variant="rasa",
)

save_skill_llm_predictions_to_csv(results, "../results/predictions/skills_qwen4b_rasa.csv")
evaluate_skill_predictions(DATASET_PATH_CV, "../results/predictions/skills_qwen4b_rasa.csv")

#### GPT-4.1 Nano

In [ ]:
results = run_llm_evaluation_skills(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=ZERO_PROMPT_SKILL,
    model_name="gpt-4.1-nano-2025-04-14",
)

save_skill_llm_predictions_to_csv(results, "../results/predictions/skills_gpt-4-1-nano_default.csv")
evaluate_skill_predictions(DATASET_PATH_CV, "../results/predictions/skills_gpt-4-1-nano_default.csv")

In [ ]:
results = run_llm_evaluation_skills(
    data_path=DATASET_PATH_CV,
    text_col="resolved_query",
    label_col="label",
    split_col="cv_fold",
    id_col="id",
    prompt=RASA_PROMPT_SKILL,
    model_name="gpt-4.1-nano-2025-04-14",
    prompt_variant="rasa",
)

save_skill_llm_predictions_to_csv(results, "../results/predictions/skills_gpt-4-1-nano_rasa.csv")
evaluate_skill_predictions(DATASET_PATH_CV, "../results/predictions/skills_gpt-4-1-nano_rasa.csv")

In [ ]:
import glob
import os
import pandas as pd

PRED_DIR = "../results/predictions"


def build_group_table(paths: list[str]) -> pd.DataFrame:
    original_counts = None
    pred_by_variant = {}

    for path in paths:
        df = pd.read_csv(path)
        variant = os.path.basename(path)[:-len(".csv")]
        if original_counts is None:
            original_counts = df["label"].value_counts()
        pred_by_variant[variant] = df["prediction"].value_counts()

    all_labels = sorted(
        set(original_counts.index).union(*[set(pc.index) for pc in pred_by_variant.values()]),
        key=str,
    )

    table = pd.DataFrame({"label_name": all_labels})
    table["original_label_count"] = [int(original_counts.get(l, 0)) for l in all_labels]
    for variant, pc in pred_by_variant.items():
        table[variant] = [int(pc.get(l, 0)) for l in all_labels]

    return table


binary_paths = sorted(glob.glob(os.path.join(PRED_DIR, "binary_*.csv")))
skill_paths = sorted(glob.glob(os.path.join(PRED_DIR, "skills_*.csv")))

binary_summary = build_group_table(binary_paths)
skill_summary = build_group_table(skill_paths)

print("=== Binary models: predicted counts per model ===")
binary_summary



In [ ]:
print("\n=== Skill-specific models: predicted counts per model ===")
skill_summary